# Transformer based approach for NLI (Task C)

#### Environment Setup

In [2]:
#pip install -q -U transformers accelerate scikit-learn pandas numpy==1.26.6 tqdm

In [3]:
import os
import random
import json
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [5]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

#### Data Loading

In [6]:
TRAIN_PATH = "data/train.csv"
DEV_PATH = "data/dev.csv"

assert os.path.exists(TRAIN_PATH), "train.csv not found in data/ folder."
assert os.path.exists(DEV_PATH), "dev.csv not found in data/ folder."

OUTPUT_DIR = "deberta_nli_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Found:", TRAIN_PATH, DEV_PATH)
print("Will save model to:", OUTPUT_DIR)

Found: train.csv dev.csv
Will save model to: deberta_nli_model


In [7]:
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)

display(train_df.head())
display(dev_df.head())

Train shape: (24432, 3)
Dev shape: (6736, 3)


,premise,hypothesis,label
0,yeah i don't know cut California in half or so...,Yeah. I'm not sure how to make that fit. Maybe...,1
1,actual names will not be used,"For the sake of privacy, actual names are not ...",1
2,The film was directed by Randall Wallace.,The film was directed by Randall Wallace and s...,1
3,"""How d'you know he'll sign me on?""Anse studie...",Anse looked at himself in a cracked mirror.,1
4,In the light of the candles his cheeks looked ...,Drew regarded his best friend and noted that i...,1


,premise,hypothesis,label
0,"By starting at the soft underbelly, the 16,000...","General Nelson A. Miles had 30,000 troops in h...",0
1,"The class had broken into a light sweat, but w...",The class grew more tense as time went on.,1
2,"Samson had his famous haircut here, but he wou...",It was unknown where exactly within the town S...,1
3,A man with a black shirt holds a baby while a ...,A darkly dressed man passes a crying baby to a...,0
4,I know that many of you are interested in addr...,The problems must be addressed,1


In [8]:
def guess_column(df: pd.DataFrame, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in cols_lower:
            return cols_lower[cand]
    return None

# Common column name variants
premise_col = guess_column(train_df, ["premise", "sentence1", "text1", "s1"])
hypothesis_col = guess_column(train_df, ["hypothesis", "sentence2", "text2", "s2"])
label_col = guess_column(train_df, ["label", "gold_label", "class", "y"])

print("Guessed columns:")
print("  premise_col   =", premise_col)
print("  hypothesis_col=", hypothesis_col)
print("  label_col     =", label_col)

if premise_col is None or hypothesis_col is None or label_col is None:
    print("\n⚠️ Could not confidently detect columns. We'll list columns next.")
    print("Columns:", list(train_df.columns))

Guessed columns:
  premise_col   = premise
  hypothesis_col= hypothesis
  label_col     = label


In [9]:
# Inspect labels
print("Train label examples:", train_df[label_col].head(10).tolist())
print("Unique train labels:", sorted(train_df[label_col].unique().tolist()))

# Create mapping
unique_labels = sorted(train_df[label_col].unique().tolist())
label2id = {lab: i for i, lab in enumerate(unique_labels)}
id2label = {i: lab for lab, i in label2id.items()}

print("label2id:", label2id)

# Convert to ids
train_df["label_id"] = train_df[label_col].map(label2id)
dev_df["label_id"]   = dev_df[label_col].map(label2id)

# Safety check: dev must not contain unseen labels
assert dev_df["label_id"].isna().sum() == 0, "Dev set contains label(s) not present in train set."

train_df["label_id"] = train_df["label_id"].astype(int)
dev_df["label_id"]   = dev_df["label_id"].astype(int)

print("num_labels:", len(label2id))

Train label examples: [1, 1, 1, 1, 1, 0, 0, 0, 1, 1]
Unique train labels: [0, 1]
label2id: {0: 0, 1: 1}
num_labels: 2


In [10]:
print("\nTrain label distribution:")
display(train_df[label_col].value_counts())

print("\nDev label distribution:")
display(dev_df[label_col].value_counts())

train_lens = (train_df[premise_col].str.split().str.len() + train_df[hypothesis_col].str.split().str.len())
dev_lens   = (dev_df[premise_col].str.split().str.len() + dev_df[hypothesis_col].str.split().str.len())

print("\nApprox total word length stats (premise+hypothesis):")
print("Train:", train_lens.describe(percentiles=[0.5, 0.9, 0.95]).to_string())
print("Dev  :", dev_lens.describe(percentiles=[0.5, 0.9, 0.95]).to_string())


Train label distribution:


,count
label,
1,12648
0,11784



Dev label distribution:


,count
label,
1,3478
0,3258



Approx total word length stats (premise+hypothesis):
Train: count    24432.000000
mean        29.363089
std         15.000247
min          2.000000
50%         27.000000
90%         47.000000
95%         54.000000
max        292.000000
Dev  : count    6736.000000
mean       29.022417
std        14.079020
min         2.000000
50%        27.000000
90%        47.000000
95%        53.000000
max       147.000000


In [11]:
# MODEL_NAME = "microsoft/deberta-v3-base"
# MODEL_NAME = "roberta-base"
MODEL_NAME = "FacebookAI/roberta-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 128
print("Model:", MODEL_NAME)
print("Max length:", MAX_LENGTH)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model: FacebookAI/roberta-large
Max length: 128


In [12]:
class NLIPairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, premise_col: str, hypothesis_col: str, label_col: str, max_length: int):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.premise_col = premise_col
        self.hypothesis_col = hypothesis_col
        self.label_col = label_col
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        prem = row[self.premise_col]
        hyp = row[self.hypothesis_col]
        label = int(row[self.label_col])

        enc = self.tokenizer(
            prem,
            hyp,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )

        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

In [13]:
train_ds = NLIPairDataset(
    train_df, tokenizer,
    premise_col=premise_col,
    hypothesis_col=hypothesis_col,
    label_col="label_id",
    max_length=MAX_LENGTH
)

dev_ds = NLIPairDataset(
    dev_df, tokenizer,
    premise_col=premise_col,
    hypothesis_col=hypothesis_col,
    label_col="label_id",
    max_length=MAX_LENGTH
)

print("Train dataset:", len(train_ds))
print("Dev dataset:", len(dev_ds))

Train dataset: 24432
Dev dataset: 6736


In [14]:
BATCH_SIZE = 16

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
dev_loader   = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Batches - train:", len(train_loader), "dev:", len(dev_loader))

Batches - train: 1527 dev: 421


In [15]:
EPOCHS = 2
LEARNING_RATE = 2e-5 # roberta
# LEARNING_RATE = 1e-5 # deberta
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_GRAD_NORM = 1.0

print("EPOCHS:", EPOCHS)
print("BATCH_SIZE:", BATCH_SIZE)
print("LR:", LEARNING_RATE)
print("WEIGHT_DECAY:", WEIGHT_DECAY)
print("WARMUP_RATIO:", WARMUP_RATIO)

EPOCHS: 2
BATCH_SIZE: 16
LR: 2e-05
WEIGHT_DECAY: 0.01
WARMUP_RATIO: 0.06


In [16]:
num_labels = len(label2id)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

model.to(device)
print("Loaded model with num_labels =", num_labels)

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded model with num_labels = 2


In [17]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

In [18]:
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("Total steps:", total_steps)
print("Warmup steps:", warmup_steps)

Total steps: 3054
Warmup steps: 183


In [19]:
@torch.no_grad()
def evaluate(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []

    for batch in tqdm(dataloader, desc="Evaluating", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(batch["labels"].cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    return acc, f1_macro, all_labels, all_preds

In [20]:
def train_one_epoch(model, dataloader, optimizer, scheduler, epoch_idx: int):
    model.train()
    running_loss = 0.0

    for step, batch in enumerate(tqdm(dataloader, desc=f"Training epoch {epoch_idx+1}", leave=False), start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item()

    avg_loss = running_loss / max(1, len(dataloader))
    return avg_loss

In [21]:
best_f1 = -1.0
best_acc = -1.0
history = []

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, epoch_idx=epoch)

    dev_acc, dev_f1, dev_labels, dev_preds = evaluate(model, dev_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Dev Acc: {dev_acc:.4f} | Dev Macro-F1: {dev_f1:.4f}")

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "dev_acc": dev_acc,
        "dev_macro_f1": dev_f1
    })

    # Save best model by macro-F1
    if dev_f1 > best_f1:
        best_f1 = dev_f1
        best_acc = dev_acc

        print("New best model found. Saving to:", OUTPUT_DIR)

        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)

        # Save training metadata for model card stuff
        metadata = {
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
            "batch_size": BATCH_SIZE,
            "epochs_trained": epoch + 1,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "warmup_ratio": WARMUP_RATIO,
            "max_grad_norm": MAX_GRAD_NORM,
            "best_dev_acc": best_acc,
            "best_dev_macro_f1": best_f1,
            "label2id": label2id,
            "id2label": id2label
        }
        with open(os.path.join(OUTPUT_DIR, "training_metadata.json"), "w") as f:
            json.dump(metadata, f, indent=2)

print("\nBest Dev Acc:", best_acc)
print("Best Dev Macro-F1:", best_f1)

Training epoch 1:   0%|          | 0/1527 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/421 [00:00<?, ?it/s]

Epoch 1/2 | Train Loss: 0.3523 | Dev Acc: 0.9111 | Dev Macro-F1: 0.9108
✅ New best model found. Saving to: deberta_nli_model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_378/3684040107.py", line 26, in <cell line: 0>
    model.save_pretrained(OUTPUT_DIR)
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 3432, in save_pretrained
    safe_save_file(shard_state_dict, filename, metadata=metadata)
  File "/usr/local/lib/python3.12/dist-packages/safetensors/torch.py", line 307, in save_file
    serialize_file(_flatten(tensors), filename, metadata=metadata)
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtraceback
    stb = value._render_traceback_()
          ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'KeyboardInterrupt' object has no att

TypeError: object of type 'NoneType' has no len()